# Example 15.2 / 1: Guidance the skill adds

Companion notebook for **15.2 AMPL Knowledge Skill for AI Agents** in the AMPL lab Documentation, section "Guidance the skill adds". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Guidance the skill adds

A usable answer has a model file with no data in it and a Python file that loads data, solves with
HiGHS, checks the status and returns a DataFrame. To check it, run both files unchanged: the
script should print `solved` for `solve_result` (or stop with an error if it is not) and the plan
6000 tons of bands and 1400 tons of coils. The pair below was written for this page to show that
result; compare the agent's files with it:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                  # steel products
param rate {PROD} > 0;     # tons per hour
param profit {PROD};       # dollars per ton
param market {PROD} >= 0;  # maximum sales, tons
param avail >= 0;          # rolling hours available

var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit, dollars

# rolling hours used fit within avail
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

products = pd.DataFrame(
    {"rate": [200, 140], "profit": [25, 30], "market": [6000, 4000]},
    index=pd.Index(["bands", "coils"], name="PROD"),
)
ampl.set_data(products, "PROD")
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
if ampl.solve_result != "solved":
    raise RuntimeError(ampl.solve_result)
print(ampl.solve_result)
print(ampl.var["Make"].to_pandas())

solved
       Make.val
bands      6000
coils      1400
